# 02 - JOB-02..07: one resumable training run (gated: B1-B12, D1-D6)

**Private compute session - not a dataset mirror.** Raw BraTS data are downloaded only from the official TCIA source into ephemeral session storage (`WORK`) and are never written to notebook output, a Kaggle Dataset, GitHub or the website. B1 basis: owner-approved alternative (amendment v1.0-A1); no TCIA authorization is claimed. Enable **GPU** and **Internet** in the notebook settings. Never paste credentials into a cell; use Kaggle Secrets if a private repository needs a token.

In [ ]:
# PARAMETERS - set before running (no secrets here)
REPO_URL = ""   # https URL of this research repository (GitHub)
COMMIT = ""     # exact commit SHA or tag to run (never a moving branch)
WORK = "/tmp/brats"                     # ephemeral session storage (raw data stay here)
EXPORT = "/kaggle/working/export"       # records/reports only - never raw data
assert REPO_URL and COMMIT, "set REPO_URL and COMMIT first"


In [ ]:
import os
os.makedirs(WORK, exist_ok=True)
os.makedirs(EXPORT, exist_ok=True)
!git clone --quiet {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo
!git checkout --quiet {COMMIT}
!git rev-parse HEAD
!pip install --quiet -e ".[io,nnunet]"
!brats-uncertainty verify-protocol

In [ ]:
JOB = ""             # JOB-02 ... JOB-07 (arm/seed from remote_compute.yaml)
DATASET_ID = None    # nnU-Net dataset id
MANIFEST_SHA256 = ""  # committed B5 manifest_sha256
SPLIT_SHA256 = ""     # committed B12 split hash
RESULTS_ROOT = "/kaggle/working/nnunet_results"  # checkpoints persist as output
STATE_ROOT = "/kaggle/working/job_state"
assert JOB and DATASET_ID and MANIFEST_SHA256 and SPLIT_SHA256

## Preconditions
1. Re-acquire the official files exactly as in notebook 01 and run `verify-inventory` against the committed B2 record: the tree must be byte-identical.
2. Build the nnU-Net raw dataset from the frozen split (dataset writer added with the B10-B12 execution; training is gated until then).
3. To resume, attach the previous session's output and copy `nnunet_results/` and `job_state/` back before running.

In [ ]:
!brats-uncertainty compute-preflight --job {JOB} --work-dir {WORK}
!brats-uncertainty job-run {JOB} --dataset-id {DATASET_ID} --results-root {RESULTS_ROOT} --state-root {STATE_ROOT} --manifest-sha256 {MANIFEST_SHA256} --split-sha256 {SPLIT_SHA256}